# Train world model

Full M3 training with inline loss curves + recon grids.

**Before training:** recollect replay (80 episodes) — the old 40-episode dump is undersized for this recipe.

Watch `kl_dyn_raw` / `kl_rep_raw`: if they stay pinned at `free_nats`, latents are not carrying detail.

Kernel → Restart, then run top-to-bottom. `RESUME = None` (architecture changed).


In [ ]:
from __future__ import annotations

import importlib
import json
import os
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import yaml
from IPython.display import clear_output, display
from torch.utils.tensorboard import SummaryWriter

import models.decoder as decoder_mod
import models.world_model as wm_mod
import training.losses as losses_mod
from models.preprocess import nchw_float_to_nhwc_uint8, nhwc_uint8_to_nchw_float
from training.device import get_device
from training.replay_buffer import ReplayBuffer, collect_random_episodes

importlib.reload(losses_mod)
importlib.reload(decoder_mod)
importlib.reload(wm_mod)
world_model_loss = losses_mod.world_model_loss
WorldModel = wm_mod.WorldModel

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

%matplotlib inline

CONFIG = Path("configs/m3_world_model.yaml")
RESUME = None  # do not resume old mushy ckpts
STEPS_OVERRIDE: int | None = None  # None = 15000 from config
RECOLLECT = True  # set False after you have the 80-episode dump

with CONFIG.open() as f:
    cfg = yaml.safe_load(f)

seed = int(cfg["seed"])
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

device = get_device()
print(f"cwd: {Path.cwd()}")
print(f"device: {device}")
print(
    f"recipe: embed={cfg['encoder']['embed_dim']} "
    f"z={cfg['rssm']['stoch']}x{cfg['rssm']['classes']} "
    f"recon_scale={cfg['train']['recon_scale']} free_nats={cfg['train']['free_nats']}"
)


## Collect / load replay


In [ ]:
replay_path = Path(cfg["collect"]["out_path"])
if RECOLLECT or not replay_path.exists():
    print(
        f"collecting {cfg['collect']['num_episodes']} episodes "
        f"(max_steps={cfg['collect']['max_episode_steps']})…"
    )
    buf = collect_random_episodes(
        env_id=str(cfg["env"]["id"]),
        num_episodes=int(cfg["collect"]["num_episodes"]),
        max_episode_steps=int(cfg["collect"]["max_episode_steps"]),
        action_dim=int(cfg["env"]["action_dim"]),
        seed=seed,
    )
    replay_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(buf.state_dict(), replay_path)
    print(f"wrote {replay_path}: episodes={len(buf)} steps={buf.num_steps}")
else:
    print(f"using existing {replay_path}")

buffer = ReplayBuffer(seed=seed)
buffer.load_state_dict(torch.load(replay_path, weights_only=False))
print(f"replay ready: episodes={len(buffer)} steps={buffer.num_steps}")


## Build model


In [ ]:
enc = cfg["encoder"]
rssm = cfg["rssm"]
dec = cfg.get("decoder", {})
heads = cfg.get("heads", {})
train_cfg = cfg["train"]

model = WorldModel.from_config_dims(
    embed_dim=int(enc["embed_dim"]),
    encoder_channels=tuple(int(c) for c in enc["channels"]),
    action_dim=int(cfg["env"]["action_dim"]),
    deter_dim=int(rssm["deter_dim"]),
    stoch=int(rssm["stoch"]),
    classes=int(rssm["classes"]),
    hidden=int(rssm["hidden"]),
    unimix=float(rssm.get("unimix", 0.01)),
    act=str(rssm.get("act", "silu")),
    initial=str(rssm.get("initial", "learned")),
    rec_depth=int(rssm.get("rec_depth", 1)),
    decoder_channels=tuple(int(c) for c in dec.get("channels", [512, 256, 128, 64])),
    head_hidden=int(heads.get("hidden", 512)),
    head_layers=int(heads.get("layers", 2)),
).to(device)

optim = torch.optim.Adam(model.parameters(), lr=float(train_cfg["lr"]))
start_step = 0
if RESUME is not None:
    ckpt = torch.load(RESUME, weights_only=False, map_location=device)
    model.load_state_dict(ckpt["model"], strict=True)
    if "optim" in ckpt:
        optim.load_state_dict(ckpt["optim"])
    start_step = int(ckpt.get("step", 0))
    print(f"resumed {RESUME} at step {start_step}")

n_params = sum(p.numel() for p in model.parameters())
print(f"params: {n_params / 1e6:.2f}M  feat_dim={model.feat_dim}")


## Train


In [ ]:
log_dir = Path(train_cfg["log_dir"])
ckpt_dir = Path(train_cfg["checkpoint_dir"])
results_dir = Path(train_cfg["results_dir"])
for p in (log_dir, ckpt_dir, results_dir):
    p.mkdir(parents=True, exist_ok=True)
writer = SummaryWriter(log_dir=str(log_dir))

steps = int(STEPS_OVERRIDE) if STEPS_OVERRIDE is not None else int(train_cfg["steps"])
batch_size = int(train_cfg["batch_size"])
seq_len = int(train_cfg["seq_len"])
log_every = int(train_cfg["log_every"])
image_every = int(train_cfg["image_every"])
ckpt_every = int(train_cfg["checkpoint_every"])

history: list[dict[str, float]] = []
last_recon_vis = None


def show_progress(history: list[dict], recon_vis) -> None:
    clear_output(wait=True)
    steps_x = [h["step"] for h in history]
    fig = plt.figure(figsize=(12, 7))

    ax0 = fig.add_subplot(2, 2, 1)
    for key, color in [
        ("total", "#000000"),
        ("recon", "#264653"),
        ("reward", "#e76f51"),
        ("continue", "#2a9d8f"),
        ("kl", "#e9c46a"),
    ]:
        ax0.plot(steps_x, [h[key] for h in history], label=key, color=color)
    ax0.set_title("loss terms")
    ax0.set_xlabel("step")
    ax0.legend(fontsize=8)

    ax1 = fig.add_subplot(2, 2, 2)
    ax1.plot(steps_x, [h["kl_dyn_raw"] for h in history], label="kl_dyn_raw", color="#e76f51")
    ax1.plot(steps_x, [h["kl_rep_raw"] for h in history], label="kl_rep_raw", color="#2a9d8f")
    ax1.axhline(float(train_cfg["free_nats"]), color="gray", linestyle="--", label="free_nats")
    ax1.set_title("KL raw (want above free_nats)")
    ax1.set_xlabel("step")
    ax1.legend(fontsize=8)

    if recon_vis is not None:
        real, pred = recon_vis
        n = min(4, real.shape[0])
        for i in range(n):
            ax = fig.add_subplot(2, 4, 5 + i)
            pair = np.concatenate([real[i].numpy(), pred[i].numpy()], axis=1)
            ax.imshow(pair)
            ax.set_title(f"real | recon  #{i}", fontsize=8)
            ax.axis("off")

    fig.tight_layout()
    display(fig)
    plt.close(fig)
    h = history[-1]
    print(
        f"step {h['step']:5d}/{start_step + steps}  total={h['total']:.4f}  "
        f"recon={h['recon']:.4f}  rew={h['reward']:.4f}  "
        f"cont={h['continue']:.4f}  kl={h['kl']:.4f} "
        f"(dyn_raw={h['kl_dyn_raw']:.3f} rep_raw={h['kl_rep_raw']:.3f})"
    )


model.train()
print(f"training {steps} steps from {start_step} on {device} …")

for step in range(start_step + 1, start_step + steps + 1):
    batch = buffer.sample(batch_size, seq_len)
    obs = batch["obs"].to(device)
    actions = batch["actions"].to(device)
    rewards = batch["rewards"].to(device)
    cont = batch["cont"].to(device)

    out = model(obs, actions)
    b, t = obs.shape[:2]
    obs_f = nhwc_uint8_to_nchw_float(obs.reshape(b * t, *obs.shape[2:])).view(b, t, 3, 64, 64)
    loss = world_model_loss(
        obs=obs_f,
        recon=out.recon,
        reward=rewards,
        reward_pred=out.reward_pred,
        cont=cont,
        cont_logit=out.cont_logit,
        post_logits=out.rssm.posterior_logits,
        prior_logits=out.rssm.prior_logits,
        unimix=model.rssm.unimix,
        dyn_scale=float(train_cfg["dyn_scale"]),
        rep_scale=float(train_cfg["rep_scale"]),
        free_nats=float(train_cfg["free_nats"]),
        recon_scale=float(train_cfg["recon_scale"]),
        reward_scale=float(train_cfg["reward_scale"]),
        continue_scale=float(train_cfg["continue_scale"]),
        kl_scale=float(train_cfg["kl_scale"]),
        recon_loss_type=str(train_cfg.get("recon_loss", "l1")),
    )

    optim.zero_grad(set_to_none=True)
    loss.total.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 100.0)
    optim.step()

    metrics = {
        "step": step,
        "total": float(loss.total.detach()),
        "recon": float(loss.recon.detach()),
        "reward": float(loss.reward.detach()),
        "continue": float(loss.continue_loss.detach()),
        "kl": float(loss.kl.detach()),
        "kl_dyn": float(loss.kl_dyn.detach()),
        "kl_rep": float(loss.kl_rep.detach()),
        "kl_dyn_raw": float(loss.kl_dyn_raw.detach()),
        "kl_rep_raw": float(loss.kl_rep_raw.detach()),
    }

    if step % log_every == 0 or step == start_step + 1:
        for k, v in metrics.items():
            if k != "step":
                writer.add_scalar(f"m3/{k}", v, step)
        history.append(metrics)
        show_progress(history, last_recon_vis)

    if step % image_every == 0 or step == start_step + 1:
        model.eval()
        with torch.no_grad():
            vis = buffer.sample(min(4, batch_size), seq_len)
            v_out = model(vis["obs"].to(device), vis["actions"].to(device))
            real = vis["obs"][:, 0].cpu()
            pred = nchw_float_to_nhwc_uint8(v_out.recon[:, 0].detach().cpu())
            last_recon_vis = (real, pred)
            from PIL import Image

            strips = [
                np.concatenate([real[i].numpy(), pred[i].numpy()], axis=1)
                for i in range(real.shape[0])
            ]
            Image.fromarray(np.concatenate(strips, axis=0), mode="RGB").save(
                results_dir / f"recon_step_{step:05d}.png"
            )
        model.train()
        if history:
            show_progress(history, last_recon_vis)

    if step % ckpt_every == 0:
        path = ckpt_dir / f"ckpt_step_{step:05d}.pt"
        torch.save(
            {"step": step, "model": model.state_dict(), "optim": optim.state_dict()},
            path,
        )
        print(f"wrote {path}")

final = ckpt_dir / "ckpt_final.pt"
torch.save(
    {
        "step": start_step + steps,
        "model": model.state_dict(),
        "optim": optim.state_dict(),
    },
    final,
)
metrics_path = results_dir / "train_metrics.json"
metrics_path.write_text(json.dumps(history, indent=2))
writer.flush()
writer.close()

show_progress(history, last_recon_vis)
print(f"\ndone. ckpt={final}  metrics={metrics_path}  tb={log_dir}")


## Final recon grid


In [ ]:
model.eval()
with torch.no_grad():
    vis = buffer.sample(6, seq_len)
    v_out = model(vis["obs"].to(device), vis["actions"].to(device))
    real = vis["obs"][:, 0].cpu()
    pred = nchw_float_to_nhwc_uint8(v_out.recon[:, 0].detach().cpu())

fig, axes = plt.subplots(6, 2, figsize=(4, 12))
for i in range(6):
    axes[i, 0].imshow(real[i].numpy())
    axes[i, 1].imshow(pred[i].numpy())
    axes[i, 0].set_ylabel(f"#{i}")
    for ax in axes[i]:
        ax.set_xticks([])
        ax.set_yticks([])
axes[0, 0].set_title("real")
axes[0, 1].set_title("recon")
fig.suptitle("Post-train reconstructions (t=0 of each window)")
fig.tight_layout()
plt.show()

from PIL import Image

out_png = results_dir / "recon_final.png"
strips = [np.concatenate([real[i].numpy(), pred[i].numpy()], axis=1) for i in range(6)]
Image.fromarray(np.concatenate(strips, axis=0), mode="RGB").save(out_png)
print(f"wrote {out_png}")
